# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# Enregistrer votre table Silver principale sous forme de vue temporaire SQL `v_silver`
SILVER_MAIN_PATH = "/lakehouse/silver/football_transfers"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1a5ad8c6-b441-4939-baa3-4f65e9f0c253;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Lignes disponibles dans v_silver : 3469314


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
df_dist_summary = spark.sql("""
SELECT COUNT(*) AS nb_indemnites_positives, AVG(transfer_fee) AS moyenne,
       STDDEV(transfer_fee) AS ecart_type, SKEWNESS(transfer_fee) AS asymetrie,
       PERCENTILE_APPROX(transfer_fee, 0.25) AS p25, PERCENTILE_APPROX(transfer_fee, 0.5) AS mediane,
       PERCENTILE_APPROX(transfer_fee, 0.75) AS p75, PERCENTILE_APPROX(transfer_fee, 0.9) AS p90,
       PERCENTILE_APPROX(transfer_fee, 0.95) AS p95, MAX(transfer_fee) AS maximum
FROM v_silver WHERE transfer_fee > 0
""")
from pyspark.sql import functions as F
df_dist_summary.withColumn("iqr", F.col("p75") - F.col("p25")).show(truncate=False)

+-----------------------+------------------+-----------------+----------------+--------+--------+---------+---------+---------+-------+---------+
|nb_indemnites_positives|moyenne           |ecart_type       |asymetrie       |p25     |mediane |p75      |p90      |p95      |maximum|iqr      |
+-----------------------+------------------+-----------------+----------------+--------+--------+---------+---------+---------+-------+---------+
|58755                  |2093145.0584971493|5642864.769475518|9.06045618116927|120000.0|450000.0|1750000.0|5000000.0|9000000.0|2.22E8 |1630000.0|
+-----------------------+------------------+-----------------+----------------+--------+--------+---------+---------+---------+-------+---------+



**Justification statistique**

Sur 58 755 indemnités positives, la moyenne atteint 2,09 millions et la médiane 450 000. L’asymétrie de 9,06 et un maximum de 222 millions montrent le poids des valeurs extrêmes. Pour décrire un transfert typique, je retiens la médiane. L’écart interquartile (IQR) de 1,63 million donne la largeur de l’intervalle qui contient la moitié centrale des montants.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [3]:
silver = spark.read.format("delta").load("/lakehouse/silver/football_transfers")
memberships = spark.read.format("delta").load("/lakehouse/bronze/club_league")
leagues = spark.read.format("delta").load("/lakehouse/bronze/leagues")
clubs = spark.read.format("delta").load("/lakehouse/bronze/clubs")
GOLD = "/lakehouse/gold"

# La saison 2022/23 couvre deux années civiles.
# On ne dispose que de l’année du transfert, donc je garde les affiliations qui désignent une seule ligue.
season = F.trim(F.col("season").cast("string"))
start = F.regexp_extract(season, r"^((?:19|20)\d{2})$", 1)
span = r"^((?:19|20)\d{2})\s*[-/]\s*((?:19|20)\d{2}|\d{2})$"
span_start = F.regexp_extract(season, span, 1)
span_end = F.regexp_extract(season, span, 2)
year_start = F.when(start != "", start).otherwise(span_start).try_cast("int")
short_end = (F.floor(year_start / 100) * 100 + span_end.try_cast("int"))
year_end = (
    F.when(start != "", year_start)
    .when(F.length(span_end) == 2, short_end + F.when(short_end < year_start, 100).otherwise(0))
    .otherwise(span_end.try_cast("int"))
)

memberships.groupBy("season").count().orderBy(F.desc("count")).show(20, truncate=False)
parsed = memberships.select(
    F.col("club_id").try_cast("bigint").alias("club_id"),
    F.nullif(F.trim(F.col("league_id").cast("string")), F.lit("")).alias("league_id"),
    year_start.alias("year_start"),
    year_end.cast("int").alias("year_end"),
)
valid = F.col("year_start").isNotNull() & F.col("year_end").between(
    F.col("year_start"), F.col("year_start") + 1
)
parsed.select(F.coalesce(valid, F.lit(False)).alias("saison_interpretable")).groupBy(
    "saison_interpretable"
).count().show()

lookup = (
    parsed.where(valid & F.col("club_id").isNotNull() & F.col("league_id").isNotNull())
    .withColumn("year", F.explode(F.sequence("year_start", "year_end")))
    .groupBy("club_id", "year")
    .agg(F.countDistinct("league_id").alias("league_count"), F.min("league_id").alias("candidate"))
    .withColumn("league_id", F.when(F.col("league_count") == 1, F.col("candidate")))
    .drop("candidate")
)
lookup.groupBy("league_count").count().orderBy("league_count").show()

assert lookup.take(1), "Aucune affiliation club–championnat exploitable."

league_names = leagues.select(
    F.nullif(F.trim(F.col("id").cast("string")), F.lit("")).alias("league_id"), "league_name", "country"
).groupBy("league_id").agg(F.min(F.struct("league_name", "country")).alias("label"))

club_names = clubs.select(
    F.col("id").try_cast("bigint").alias("club_id"), "club_name"
).groupBy("club_id").agg(F.min("club_name").alias("origin_club"))

transfers = silver.select(
    F.col("year").try_cast("int").alias("year"),
    F.col("left_club_id").try_cast("bigint").alias("origin_club_id"),
    F.col("joined_club_id").try_cast("bigint").alias("destination_club_id"),
    "player_name", F.col("club_name").alias("destination_club"), "transfer_type",
    F.when((F.col("transfer_fee") >= 0) & (F.col("transfer_fee") < float("inf")),
           F.col("transfer_fee")).alias("fee"),
)
expected = transfers.count()
for side in ("origin", "destination"):
    by_side = lookup.join(F.broadcast(league_names), "league_id", "left").select(
        F.col("club_id").alias(f"{side}_club_id"), "year",
        F.col("league_id").alias(f"{side}_league_id"),
        F.col("label.league_name").alias(f"{side}_league"),
        F.col("label.country").alias(f"{side}_country"),
        F.when(F.col("league_count") > 1, "ambigu")
        .when(F.col("league_id").isNotNull(), "rattaché")
        .otherwise("inconnu").alias(f"{side}_status"),
    )
    transfers = transfers.join(by_side, ["year", f"{side}_club_id"], "left").fillna(
        {f"{side}_status": "inconnu"}
    )

transfers = transfers.join(
    F.broadcast(club_names.withColumnRenamed("club_id", "origin_club_id")), "origin_club_id", "left"
).cache()

assert transfers.count() == expected, "La jointure championnat a changé le nombre de lignes."
coverage = transfers.groupBy("year", "origin_status", "destination_status").count()
coverage.groupBy("origin_status", "destination_status").sum("count").show(truncate=False)

transfers.createOrReplaceTempView("football_transfers_enriched")

GOLD_PARETO_PATH = f"{GOLD}/football_league_pareto"
df_gold_1 = spark.sql("""
WITH leagues AS (
    SELECT year, destination_league_id AS league_id,
           MIN(CONCAT_WS(' · ', destination_league, destination_country)) AS league,
           COUNT(*) AS transfers, COUNT(fee) AS known_fees, SUM(fee) AS total,
           PERCENTILE_APPROX(IF(fee > 0, fee, NULL), 0.50) AS median_fee,
           PERCENTILE_APPROX(IF(fee > 0, fee, NULL), 0.75)
             - PERCENTILE_APPROX(IF(fee > 0, fee, NULL), 0.25) AS iqr
    FROM football_transfers_enriched
    WHERE year IS NOT NULL AND destination_league_id IS NOT NULL
    GROUP BY year, destination_league_id
)
SELECT *, DENSE_RANK() OVER (PARTITION BY year ORDER BY total DESC NULLS LAST) AS spend_rank,
       ROW_NUMBER() OVER (PARTITION BY year ORDER BY total DESC NULLS LAST, league_id) AS pareto_order,
       TRY_DIVIDE(SUM(total) OVER (PARTITION BY year ORDER BY total DESC NULLS LAST, league_id
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
         SUM(total) OVER (PARTITION BY year)) AS cumulative_share
FROM leagues
""")
df_gold_1.coalesce(1).write.format("delta").mode("overwrite").save(GOLD_PARETO_PATH)
df_gold_1.where("year = 2023").orderBy("pareto_order").show(12, truncate=False)

+------+-----+
|season|count|
+------+-----+
|2023  |6848 |
|2022  |6480 |
|2021  |6342 |
|2020  |5953 |
|2019  |5674 |
|2018  |5397 |
|2017  |5129 |
|2016  |4899 |
|2024  |4859 |
|2015  |4580 |
|2014  |4324 |
|2013  |4012 |
|2012  |3742 |
|2011  |3272 |
|2010  |3020 |
|2009  |2605 |
|2008  |2315 |
|2007  |2105 |
|2006  |1977 |
|2005  |1891 |
+------+-----+
only showing top 20 rows
+--------------------+------+
|saison_interpretable| count|
+--------------------+------+
|                true|127962|
|               false|    18|
+--------------------+------+

+------------+------+
|league_count| count|
+------------+------+
|           1|121792|
|           2|  2708|
|           3|   235|
|           4|    11|
|           5|     1|
+------------+------+



+-------------+------------------+----------+
|origin_status|destination_status|sum(count)|
+-------------+------------------+----------+
|inconnu      |ambigu            |22618     |
|inconnu      |rattaché          |649805    |
|rattaché     |rattaché          |757798    |
|ambigu       |rattaché          |13654     |
|ambigu       |ambigu            |19673     |
|ambigu       |inconnu           |21494     |
|rattaché     |ambigu            |13116     |
|rattaché     |inconnu           |654446    |
|inconnu      |inconnu           |1316710   |
+-------------+------------------+----------+



+----+---------+-----------------------------------+---------+----------+-------------+-----------+---------+----------+------------+-------------------+
|year|league_id|league                             |transfers|known_fees|total        |median_fee |iqr      |spend_rank|pareto_order|cumulative_share   |
+----+---------+-----------------------------------+---------+----------+-------------+-----------+---------+----------+------------+-------------------+
|2023|GB1      |Premier League · England           |346      |346       |2.928039999E9|1.6149999E7|2.835E7  |1         |1           |0.28189318345684694|
|2023|FR1      |Ligue 1 · France                   |269      |269       |1.17872E9    |4000000.0  |9700000.0|2         |2           |0.3953728945665771 |
|2023|IT1      |Serie A · Italy                    |384      |384       |1.00387E9    |3000000.0  |6500000.0|3         |3           |0.49201915287366005|
|2023|SA1      |Saudi Pro League · Saudi Arabia    |336      |336       |9.7

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:
GOLD_TIME_PATH = f"{GOLD}/football_league_annual"
df_gold_2 = spark.sql("""
WITH annual AS (
    SELECT year,
           CASE WHEN origin_league_id = destination_league_id THEN 'Même championnat'
                ELSE 'Entre championnats' END AS scope,
           COUNT(*) AS transfers, COUNT(fee) AS known_fees,
           AVG(IF(fee > 0, fee, NULL)) AS mean_fee,
           PERCENTILE_APPROX(IF(fee > 0, fee, NULL), 0.50) AS median_fee
    FROM football_transfers_enriched
    WHERE year IS NOT NULL AND origin_league_id IS NOT NULL AND destination_league_id IS NOT NULL
    GROUP BY 1, 2
), history AS (
    SELECT *, LAG(year) OVER (PARTITION BY scope ORDER BY year) AS previous_year,
           LAG(transfers) OVER (PARTITION BY scope ORDER BY year) AS previous_transfers
    FROM annual
)
SELECT *, IF(year = previous_year + 1,
           TRY_DIVIDE(transfers - previous_transfers, previous_transfers), NULL) AS growth
FROM history
""")
df_gold_2.coalesce(1).write.format("delta").mode("overwrite").save(GOLD_TIME_PATH)
df_gold_2.where("year >= 2019").orderBy("year", "scope").show(truncate=False)

+----+------------------+---------+----------+------------------+----------+-------------+------------------+---------------------+
|year|scope             |transfers|known_fees|mean_fee          |median_fee|previous_year|previous_transfers|growth               |
+----+------------------+---------+----------+------------------+----------+-------------+------------------+---------------------+
|2019|Entre championnats|29269    |29269     |4009081.7343550446|850000.0  |2018         |28115             |0.04104570513960519  |
|2019|Même championnat  |14653    |14653     |3973069.4915254237|950000.0  |2018         |14540             |0.007771664374140302 |
|2020|Entre championnats|28765    |28765     |3114664.2604166665|550000.0  |2019         |29269             |-0.017219583860056717|
|2020|Même championnat  |14477    |14477     |2494885.3754940713|500000.0  |2019         |14653             |-0.012011192247321367|
|2021|Entre championnats|33423    |33423     |2742572.4206128134|600000.0  |

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [5]:
# Pour compter un flux, il faut avoir identifié les deux ligues.
flows = spark.sql("""
WITH pairs AS (
    SELECT year, origin_league_id, destination_league_id,
           MIN(origin_league) AS origin_league, MIN(origin_country) AS origin_country,
           MIN(destination_league) AS destination_league, MIN(destination_country) AS destination_country,
           COUNT(*) AS transfers, COUNT(fee) AS known_fees,
           COUNT_IF(fee > 0) AS positive_fees, SUM(fee) AS known_total,
           AVG(IF(fee > 0, fee, NULL)) AS mean_positive_fee,
           PERCENTILE_APPROX(IF(fee > 0, fee, NULL), 0.50) AS median_positive_fee
    FROM football_transfers_enriched
    WHERE year IS NOT NULL AND origin_league_id IS NOT NULL AND destination_league_id IS NOT NULL
    GROUP BY year, origin_league_id, destination_league_id
)
SELECT *, ROW_NUMBER() OVER (PARTITION BY year, destination_league_id
                             ORDER BY transfers DESC, origin_league_id) AS origin_rank
FROM pairs
""")
coverage = spark.sql("""
SELECT year, origin_status, destination_status, COUNT(*) AS count
FROM football_transfers_enriched GROUP BY year, origin_status, destination_status
""")
details = spark.sql("""
SELECT year, origin_league_id, destination_league_id, player_name,
       origin_club, destination_club, transfer_type, fee
FROM football_transfers_enriched
WHERE year IS NOT NULL AND origin_league_id IS NOT NULL AND destination_league_id IS NOT NULL
""")
for name, frame in {"flows": flows.coalesce(1), "coverage": coverage.coalesce(1), "transfers": details, "enriched": transfers}.items():
    frame.write.format("delta").mode("overwrite").save(f"{GOLD}/football_league_{name}")
flows.where("year = 2023 AND origin_rank = 1").orderBy(F.desc("transfers")).show(10, truncate=False)
transfers.unpersist()
# La compaction réduit les métadonnées de petits fichiers gérées par le NameNode.
SILVER_PATH = SILVER_MAIN_PATH
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)

from delta.tables import DeltaTable

gold_tables = {
    name: spark.read.format("delta").load(f"{GOLD}/football_league_{name}")
    for name in ("pareto", "annual", "flows", "coverage", "transfers", "enriched")
}
sizes = {name: frame.count() for name, frame in gold_tables.items()}
print("Nombre de lignes par table Gold :", sizes)
assert all(sizes.values()), "Une table Gold est vide."
assert sizes["enriched"] == expected, "Le nombre de transferts a changé."
coverage_total = gold_tables["coverage"].agg(F.sum("count")).first()[0]
flows_total = gold_tables["flows"].agg(F.sum("transfers")).first()[0]
annual_total = gold_tables["annual"].agg(F.sum("transfers")).first()[0]
assert coverage_total == expected, "Couverture incohérente avec Silver."
assert flows_total == annual_total == sizes["transfers"], "Effectifs Gold incompatibles."
matched_count = gold_tables["coverage"].where(
    (F.col("origin_status") == "rattaché") & (F.col("destination_status") == "rattaché")
).agg(F.sum("count")).first()[0]
assert matched_count == flows_total, "Les flux doivent couvrir exactement les transferts doublement rattachés."
print(f"Couverture globale : {flows_total:,} / {expected:,} ({flows_total / expected:.1%}).")

(
    gold_tables["coverage"].where(F.col("year") >= 2019).groupBy("year")
    .agg(
        F.sum("count").alias("recenses"),
        F.sum(F.when((F.col("origin_status") == "rattaché") &
                     (F.col("destination_status") == "rattaché"), F.col("count")).otherwise(0))
        .alias("deux_ligues"),
    )
    .withColumn("couverture_pct", F.round(100 * F.try_divide("deux_ligues", "recenses"), 2))
    .orderBy("year").show(30)
)
print("Principaux flux en 2023 :")
(
    gold_tables["flows"].where(F.col("year") == 2023)
    .select("origin_league", "destination_league", "transfers", "known_fees")
    .orderBy(F.desc("transfers"), "origin_league", "destination_league").show(5, truncate=False)
)
pareto_2023 = gold_tables["pareto"].where(F.col("year") == 2023)
threshold = pareto_2023.where(F.col("cumulative_share") >= 0.8).agg(F.min("pareto_order")).first()[0]
if threshold is not None:
    print(f"Pareto 2023 : les {threshold} premiers championnats atteignent 80 % des montants connus.")
print("Dernières transactions Delta Silver :")
DeltaTable.forPath(spark, SILVER_PATH).history(3).select("version", "operation").show()
print("Contrôles de cohérence Gold réussis.")

+----+----------------+---------------------+------------------------------------+--------------+------------------------------------+-------------------+---------+----------+-------------+-----------+------------------+-------------------+-----------+
|year|origin_league_id|destination_league_id|origin_league                       |origin_country|destination_league                  |destination_country|transfers|known_fees|positive_fees|known_total|mean_positive_fee |median_positive_fee|origin_rank|
+----+----------------+---------------------+------------------------------------+--------------+------------------------------------+-------------------+---------+----------+-------------+-----------+------------------+-------------------+-----------+
|2023|Y246            |Y047                 |YFL U16 (2024)                      |Russia        |YFL U17 (2024)                      |Russia             |620      |620       |0            |0.0        |NULL              |NULL               |1

+----+--------+-----------+--------------+
|year|recenses|deux_ligues|couverture_pct|
+----+--------+-----------+--------------+
|2019|  183484|      43922|         23.94|
|2020|  175420|      43242|         24.65|
|2021|  196402|      50122|         25.52|
|2022|  205866|      52865|         25.68|
|2023|  202275|      55825|          27.6|
|2024|   64753|      20281|         31.32|
|2025|     167|          0|           0.0|
|2026|      16|          0|           0.0|
|2027|       3|          0|           0.0|
+----+--------+-----------+--------------+

Principaux flux en 2023 :
+-----------------------+-----------------------+---------+----------+
|origin_league          |destination_league     |transfers|known_fees|
+-----------------------+-----------------------+---------+----------+
|YFL U16 (2024)         |YFL U17 (2024)         |620      |620       |
|YFL U17 (2024)         |U19 M-Liga (since 2024)|394      |394       |
|2. Division B          |2. Division B          |290      |